In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
url="https://docs.google.com/spreadsheets/d/1N4-eLmaH5I2oNfUTh-qppwjMmC0Ktqhh/export?format=csv&gid=727130089"
df=pd.read_csv(url)
print(df.shape)
print(df.head())
print(df.info())

(1200, 14)
     OrderID        Date CustomerID  Product  Quantity  UnitPrice  \
0  ORD200000  2023-01-04     C72649  Monitor         5     570.62   
1  ORD200001  2024-08-23     C75739    Phone         2     151.35   
2  ORD200002  2024-02-27     C81728   Tablet         5     550.68   
3  ORD200003  2023-10-15     C33540    Chair         1     273.19   
4  ORD200004  2025-05-08     C81840  Printer         4     626.01   

  ShippingAddress PaymentMethod OrderStatus TrackingNumber  ItemsInCart  \
0     928 Main St    Debit Card     Shipped    TRK37947903            7   
1     823 Main St        Online     Shipped    TRK91186779            3   
2     512 Main St   Credit Card   Cancelled    TRK42903982            8   
3     275 Main St    Debit Card    Returned    TRK62788070            5   
4     668 Main St        Online   Delivered    TRK29241424            8   

  CouponCode ReferralSource  TotalPrice  
0     SAVE10      Instagram     2853.10  
1     SAVE10       Referral      302.70

In [2]:
common_place_hold=["-","NaN","?","Null","NULL","na",""]
for col in df.columns:
  if df[col].isin(common_place_hold).any():
    df[col].replace(common_place_hold,pd.NA,inplace=True)


In [3]:
missing_perc=df.isnull().mean()*100
print(missing_perc)

OrderID             0.00
Date                0.00
CustomerID          0.00
Product             0.00
Quantity            0.00
UnitPrice           0.00
ShippingAddress     0.00
PaymentMethod       0.00
OrderStatus         0.00
TrackingNumber      0.00
ItemsInCart         0.00
CouponCode         25.75
ReferralSource      0.00
TotalPrice          0.00
dtype: float64


In [4]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
low_missing_per = missing_perc[missing_perc < 5].index
mid_missing_per = missing_perc[(missing_perc >= 5) & (missing_perc <= 20)].index
high_missing_per = missing_perc[(missing_perc > 20)].index
df=df.dropna(subset = low_missing_per)
for col in numeric_cols:
  if col in low_missing_per:
    overall_mean = df[col].median()
    df[col] = df[col].fillna(overall_mean)


In [5]:

knn_cols = [col for col in high_missing_per if col in numeric_cols]

if len(knn_cols) >0:
  scaler = StandardScaler()
  imputer = KNNImputer(n_neighbors=5)
  df_knn = df[knn_cols]
  df_scaled = scaler.fit_transform(df_knn)
  df_imputed = imputer.fit_transform(df_scaled)
  df_result = scaler.inverse_transform(df_imputed)
  df[knn_cols] = df_result

In [6]:
for col in numeric_cols:
  Q1=df[col].quantile(0.25)
  Q3=df[col].quantile(0.75)
  IQR=Q3-Q1
  lower_bound=Q1-1.5*IQR
  upper_bound=Q3+1.5*IQR
  outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
  print(f"{col}: lower={lower_bound:.2f}, upper={upper_bound:.2f}, outliers={len(outliers)}")

Quantity: lower=-1.00, upper=7.00, outliers=0
UnitPrice: lower=-317.20, upper=1024.83, outliers=0
ItemsInCart: lower=-0.50, upper=11.50, outliers=0
TotalPrice: lower=-1341.41, upper=3330.41, outliers=8


In [ ]:
# Already have these values from your IQR loop, just apply clip
upper_total = df["TotalPrice"].quantile(0.75) + 1.5 * (df["TotalPrice"].quantile(0.75) - df["TotalPrice"].quantile(0.25))
lower_total = df["TotalPrice"].quantile(0.25) - 1.5 * (df["TotalPrice"].quantile(0.75) - df["TotalPrice"].quantile(0.25))

df["TotalPrice"] = np.clip(df["TotalPrice"], lower_total, upper_total)

# Verify
print(df["TotalPrice"].describe())
print("Outliers remaining:", len(df[(df["TotalPrice"] < lower_total) | (df["TotalPrice"] > upper_total)]))

In [7]:
df["Revenue"] = df["TotalPrice"] / df["Quantity"]
print(df["Revenue"].head())

0    570.62
1    151.35
2    550.68
3    273.19
4    626.01
Name: Revenue, dtype: float64


In [8]:
df["Date"] = pd.to_datetime(df["Date"])
df["OrderMonth"] = df["Date"].dt.month
df["OrderYear"] = df["Date"].dt.year
df["OrderDay"] = df["Date"].dt.dayofweek

In [13]:
print(df["PaymentMethod"].nunique())   # how many unique products?
print(df["PaymentMethod"].unique())    # what are they
df = pd.get_dummies(df, columns=["PaymentMethod"], prefix="PaymentMethod")

KeyError: 'PaymentMethod'

In [14]:
print(df.columns)

Index(['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice',
       'ShippingAddress', 'OrderStatus', 'TrackingNumber', 'ItemsInCart',
       'CouponCode', 'ReferralSource', 'TotalPrice', 'Revenue', 'OrderMonth',
       'OrderYear', 'OrderDay', 'PaymentMethod_Cash',
       'PaymentMethod_Credit Card', 'PaymentMethod_Debit Card',
       'PaymentMethod_Gift Card', 'PaymentMethod_Online'],
      dtype='object')
